In [2]:
# !pip install -r requirements.txt

In [3]:
import sys

repo_root = "/serafin/pcelayes/repos/sna_classifier"

sys.path.append(repo_root) # go to parent dir


In [4]:
from tw_dataset.settings import PROJECT_PATH, GT_GRAPH_PATH, NX_GRAPH_PATH, IG_GRAPH_PATH, DATASETS_FOLDER

In [5]:
import networkx as nx

In [6]:
graph = nx.read_graphml(IG_GRAPH_PATH)

In [9]:
import pandas as pd

G = graph
df = pd.DataFrame.from_dict(G.nodes(data=True), orient="index")  # wrong — use:
df = pd.DataFrame(dict(G.nodes(data=True))).T
# or more cleanly:
df = pd.DataFrame([{"node": n, **attrs} for n, attrs in G.nodes(data=True)])

AttributeError: 'NodeDataView' object has no attribute 'values'

In [10]:
first_node = next(iter(G.nodes))
print(G.nodes[first_node].keys())

dict_keys(['twid'])


In [6]:
len(graph.nodes())

5589

In [7]:
len(graph.edges())

261005

In [8]:
graph

In [9]:
# !pip install torch

In [11]:
!pip install torch-geometric-signed-directed

  Using cached torch_geometric_signed_directed-1.1.1-py3-none-any.whl.metadata (27 kB)
  Using cached torch_geometric-2.7.0-py3-none-any.whl.metadata (63 kB)
  Using cached xxhash-3.6.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (13 kB)
Using cached torch_geometric_signed_directed-1.1.1-py3-none-any.whl (119 kB)
Using cached torch_geometric-2.7.0-py3-none-any.whl (1.3 MB)
Using cached xxhash-3.6.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (193 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torch-geometric-signed-directed]ic]


In [12]:
import torch
import networkx as nx
import numpy as np
from torch_geometric.utils import from_networkx, degree
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def prepare_graph(G: nx.DiGraph):
    """
    Convert a NetworkX DiGraph to PyG format with structural node features.
    """
    logger.info("Starting graph preparation")
    logger.info(f"Input graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    
    # --- 1. Convert to PyG ---
    logger.info("Step 1: Converting NetworkX graph to PyTorch Geometric format")
    data = from_networkx(G)
    edge_index = data.edge_index  # [2, 260k]
    N = G.number_of_nodes()
    logger.info(f"Converted to edge_index with shape: {edge_index.shape}")

    # --- 2. Build structural node features ---
    logger.info("Step 2: Building structural node features")
    
    logger.info("Computing in-degree and out-degree")
    in_deg  = degree(edge_index[1], num_nodes=N)   # in-degree
    out_deg = degree(edge_index[0], num_nodes=N)   # out-degree
    total_deg = in_deg + out_deg
    logger.info(f"Degree statistics - Mean in-degree: {in_deg.mean():.2f}, Mean out-degree: {out_deg.mean():.2f}")

    # reciprocity per node: fraction of neighbors with mutual edges
    logger.info("Computing reciprocity per node")
    reciprocal_edges = sum(1 for u, v in G.edges() if G.has_edge(v, u))
    logger.info(f"Total reciprocal edges: {reciprocal_edges}")
    recip = torch.tensor(
        [sum(1 for nb in G.successors(n) if G.has_edge(nb, n)) / max(G.out_degree(n), 1)
         for n in G.nodes()],
        dtype=torch.float
    )
    logger.info(f"Mean reciprocity: {recip.mean():.4f}")

    # in/out degree ratio (direction bias per node)
    logger.info("Computing in/out degree ratio")
    ratio = in_deg / (total_deg + 1e-8)
    logger.info(f"Mean degree ratio: {ratio.mean():.4f}")

    # local clustering coefficient (undirected view)
    # logger.info("Computing local clustering coefficient")
    # Note: too slow!
    # clustering = torch.tensor(
    #     [nx.clustering(G.to_undirected(), n) for n in G.nodes()],
    #     dtype=torch.float
    # )
    # logger.info(f"Mean clustering coefficient: {clustering.mean():.4f}")

    # --- 3. Stack and normalize ---
    logger.info("Step 3: Stacking and normalizing features")
    x = torch.stack([
        in_deg,
        out_deg,
        total_deg,
        ratio,        # 0 = source-like, 1 = sink-like
        recip,        # 0 = no reciprocal edges, 1 = all reciprocal
        # clustering,
    ], dim=-1)  # [N, 6]
    logger.info(f"Feature matrix shape: {x.shape}")

    # z-score normalize each feature
    logger.info("Applying z-score normalization")
    mean = x.mean(dim=0, keepdim=True)
    std  = x.std(dim=0, keepdim=True) + 1e-8
    x = (x - mean) / std
    logger.info(f"Normalized feature statistics - Mean: {x.mean(dim=0)}, Std: {x.std(dim=0)}")

    logger.info("Graph preparation completed successfully")
    return edge_index, x, N

In [13]:
import torch
import torch.nn.functional as F
from torch_geometric_signed_directed.nn.directed import MagNetConv
from torch_geometric.utils import negative_sampling

class MagNetUnsupervised(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, q=0.25, K=1):
        super().__init__()
        self.conv1 = MagNetConv(in_channels, hidden_channels, trainable_q=False, q=q, K=K)
        self.conv2 = MagNetConv(hidden_channels, hidden_channels, trainable_q=False, q=q, K=K)

    def encode(self, x, edge_index):
        x_r, x_i = self.conv1(x, x, edge_index)
        x_r, x_i = F.relu(x_r), F.relu(x_i)
        x_r, x_i = self.conv2(x_r, x_i, edge_index)
        # concatenate real and imaginary parts
        return torch.cat([x_r, x_i], dim=-1)  # [N, 2*hidden]

    def decode(self, z, edge_index):
        # dot product between source and target node embeddings
        return (z[edge_index[0]] * z[edge_index[1]]).sum(dim=-1)

    def forward(self, x, edge_index):
        z = self.encode(x, edge_index)
        # positive edges
        pos_scores = self.decode(z, edge_index)
        # negative samples (random non-edges)
        neg_edge_index = negative_sampling(edge_index, num_nodes=x.size(0),
                                           num_neg_samples=edge_index.size(1))
        neg_scores = self.decode(z, neg_edge_index)
        return pos_scores, neg_scores

In [14]:
edge_index, x, N = prepare_graph(graph)

2026-04-24 12:01:45,097 - INFO - Starting graph preparation
2026-04-24 12:01:45,100 - INFO - Input graph: 5589 nodes, 261005 edges
2026-04-24 12:01:45,100 - INFO - Step 1: Converting NetworkX graph to PyTorch Geometric format
2026-04-24 12:01:46,853 - INFO - Converted to edge_index with shape: torch.Size([2, 261005])
2026-04-24 12:01:46,855 - INFO - Step 2: Building structural node features
2026-04-24 12:01:46,855 - INFO - Computing in-degree and out-degree
2026-04-24 12:01:46,888 - INFO - Degree statistics - Mean in-degree: 46.70, Mean out-degree: 46.70
2026-04-24 12:01:46,892 - INFO - Computing reciprocity per node
2026-04-24 12:01:46,972 - INFO - Total reciprocal edges: 53649
2026-04-24 12:01:47,035 - INFO - Mean reciprocity: 0.2072
2026-04-24 12:01:47,036 - INFO - Computing in/out degree ratio
2026-04-24 12:01:47,036 - INFO - Mean degree ratio: 0.3669
2026-04-24 12:01:47,036 - INFO - Step 3: Stacking and normalizing features
2026-04-24 12:01:47,037 - INFO - Feature matrix shape: to

In [15]:
# Training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [16]:
device

device(type='cuda')

In [20]:
model = MagNetUnsupervised(in_channels=5, hidden_channels=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.999)

x = x.to(device)
edge_index = edge_index.to(device)

for epoch in range(1000):
    model.train()
    optimizer.zero_grad()
    pos_scores, neg_scores = model(x, edge_index)
    loss = F.binary_cross_entropy_with_logits(
        torch.cat([pos_scores, neg_scores]),
        torch.cat([torch.ones(pos_scores.size(0), device=device),
                   torch.zeros(neg_scores.size(0), device=device)])
    )
    loss.backward()
    optimizer.step()
    scheduler.step()

    print(f"Epoch {epoch + 1}, Loss: {loss.item():.4f}, LR: {scheduler.get_last_lr()[0]:.6f}")

# Extract embeddings
model.eval()
with torch.no_grad():
    embeddings = model.encode(x, edge_index)  # [5600, 128]


Epoch 1, Loss: 11.6499, LR: 0.099900
Epoch 2, Loss: 67.9823, LR: 0.099800
Epoch 3, Loss: 97.5366, LR: 0.099700
Epoch 4, Loss: 27.3673, LR: 0.099601
Epoch 5, Loss: 18.5043, LR: 0.099501
Epoch 6, Loss: 15.5230, LR: 0.099401
Epoch 7, Loss: 6.9887, LR: 0.099302
Epoch 8, Loss: 5.2664, LR: 0.099203
Epoch 9, Loss: 8.0691, LR: 0.099104
Epoch 10, Loss: 7.3364, LR: 0.099004
Epoch 11, Loss: 6.3032, LR: 0.098905
Epoch 12, Loss: 4.4983, LR: 0.098807
Epoch 13, Loss: 4.8054, LR: 0.098708
Epoch 14, Loss: 4.4258, LR: 0.098609
Epoch 15, Loss: 4.0719, LR: 0.098510
Epoch 16, Loss: 3.3289, LR: 0.098412
Epoch 17, Loss: 3.1035, LR: 0.098314
Epoch 18, Loss: 2.4234, LR: 0.098215
Epoch 19, Loss: 2.3340, LR: 0.098117
Epoch 20, Loss: 1.9734, LR: 0.098019
Epoch 21, Loss: 1.6146, LR: 0.097921
Epoch 22, Loss: 1.4108, LR: 0.097823
Epoch 23, Loss: 1.2379, LR: 0.097725
Epoch 24, Loss: 1.2189, LR: 0.097627
Epoch 25, Loss: 1.1439, LR: 0.097530
Epoch 26, Loss: 1.0723, LR: 0.097432
Epoch 27, Loss: 1.0512, LR: 0.097335
Epoc

In [21]:
torch.save(embeddings, 'node_embeddings.pt')

In [ ]:
# np.save('node_embeddings.npy', embeddings.numpy())